<a href="https://colab.research.google.com/github/Dilshan-Prince/Statistical-Learning-e21100/blob/main/Assignment4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
"""
data_inspector.py
=================
Reusable Python classes for automated CSV data ingestion, cleaning,
feature engineering preparation, and interactive statistical visualization.

Classes
-------
DataInspector  – core EDA / cleaning / normalisation engine
PlottingMethods – granular chart helpers returning HTML-embeddable figures

Usage (Google Colab)
--------------------
    from data_inspector import DataInspector, PlottingMethods
    di = DataInspector()
    di.upload_data()          # triggers Colab file picker
    di.data_summary()
    di.handle_missing_values(strategy="median")
    di.plot_numeric_univariate("Age")
    di.plot_all_associations_heatmap()
"""

from __future__ import annotations

import io
import warnings
from typing import Dict, List, Optional, Tuple, Union

import numpy as np
import pandas as pd
from scipy import stats

# ── optional imports (graceful degradation) ──────────────────────────────────
try:
    import plotly.express as px
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
    PLOTLY_AVAILABLE = True
except ImportError:
    PLOTLY_AVAILABLE = False
    warnings.warn("plotly not installed – visualisation methods will be unavailable.")

try:
    from sklearn.preprocessing import (
        MinMaxScaler, StandardScaler, RobustScaler,
        OrdinalEncoder, OneHotEncoder,
    )
    SKLEARN_AVAILABLE = True
except ImportError:
    SKLEARN_AVAILABLE = False
    warnings.warn("scikit-learn not installed – normalisation methods will be unavailable.")

# ── garbage strings treated as NaN everywhere ─────────────────────────────────
_GARBAGE = {"?", "n/a", "na", "null", "none", "nan", "", " ", "--", "n.a.", "n.a", "unknown"}


# =============================================================================
# Helper utilities
# =============================================================================

def _cramers_v(x: pd.Series, y: pd.Series) -> float:
    """Cramér's V association statistic for two categorical series."""
    confusion = pd.crosstab(x, y)
    chi2 = stats.chi2_contingency(confusion, correction=False)[0]
    n = confusion.values.sum()
    phi2 = chi2 / n
    r, k = confusion.shape
    phi2corr = max(0.0, phi2 - ((k - 1) * (r - 1)) / (n - 1))
    rcorr = r - (r - 1) ** 2 / (n - 1)
    kcorr = k - (k - 1) ** 2 / (n - 1)
    denom = min(kcorr - 1, rcorr - 1)
    return 0.0 if denom <= 0 else np.sqrt(phi2corr / denom)


def _eta_squared(cat: pd.Series, num: pd.Series) -> float:
    """η² (Eta-squared) via one-way ANOVA – effect size for cat→num."""
    groups = [num[cat == lvl].dropna() for lvl in cat.unique()]
    groups = [g for g in groups if len(g) > 0]
    if len(groups) < 2:
        return 0.0
    f, p = stats.f_oneway(*groups)
    if np.isnan(f):
        return 0.0
    grand_mean = num.mean()
    ss_between = sum(len(g) * (g.mean() - grand_mean) ** 2 for g in groups)
    ss_total = ((num - grand_mean) ** 2).sum()
    return 0.0 if ss_total == 0 else ss_between / ss_total


# =============================================================================
# DataInspector
# =============================================================================

class DataInspector:
    """
    End-to-end tool for CSV data ingestion, cleaning, feature engineering
    preparation, and interactive statistical visualisation.

    Attributes
    ----------
    df : pd.DataFrame
        The working dataset (mutated in-place by cleaning methods).
    _original_df : pd.DataFrame
        Immutable snapshot taken at load time.
    """

    # ── construction ──────────────────────────────────────────────────────────

    def __init__(self) -> None:
        self.df: Optional[pd.DataFrame] = None
        self._original_df: Optional[pd.DataFrame] = None

    # ── 1. DATA INGESTION & SANITISATION ──────────────────────────────────────

    def upload_data(self, filepath: Optional[str] = None) -> None:
        """
        Load a CSV into ``self.df``.

        Parameters
        ----------
        filepath : str, optional
            Absolute or relative path to a CSV file.  When *None* and the
            environment is Google Colab, the Colab file-picker widget is
            launched automatically.

        Side-effects
        ------------
        * Replaces garbage strings with ``np.nan``.
        * Force-converts columns to numeric where possible.
        * Stores an immutable copy in ``self._original_df``.
        """
        if filepath is None:
            try:
                from google.colab import files  # type: ignore
                print("📂  Please select a CSV file …")
                uploaded = files.upload()
                name = next(iter(uploaded))
                raw = io.BytesIO(uploaded[name])
            except (ImportError, StopIteration):
                raise ValueError(
                    "No filepath supplied and not running in Colab. "
                    "Pass filepath='/path/to/file.csv'."
                )
        else:
            raw = filepath

        df = pd.read_csv(raw, na_values=list(_GARBAGE), keep_default_na=True)

        # secondary pass: lower-strip strings not caught by na_values
        for col in df.select_dtypes(include="object").columns:
            df[col] = df[col].apply(
                lambda v: np.nan
                if isinstance(v, str) and v.strip().lower() in _GARBAGE
                else v
            )

        # auto-type correction
        df = self._auto_type_correct(df)

        self.df = df
        self._original_df = df.copy(deep=True)
        print(f"✅  Loaded {df.shape[0]:,} rows × {df.shape[1]} columns.")

    @staticmethod
    def _auto_type_correct(df: pd.DataFrame) -> pd.DataFrame:
        """
        Attempt to coerce object columns to numeric.  A column is only
        converted if the result is **not** entirely null (preserves columns
        that are genuinely categorical).
        """
        for col in df.select_dtypes(include="object").columns:
            trial = pd.to_numeric(df[col], errors="coerce")
            if trial.notna().any():
                df[col] = trial
        return df

    # ── 2. STRUCTURAL ANALYSIS & CLEANING ─────────────────────────────────────

    def data_summary(self) -> None:
        """
        Print a concise structural report:
        * row / column counts
        * first 20 rows preview
        * numerical vs categorical breakdown
        * per-column null counts and dtypes
        """
        self._require_data()
        df = self.df

        print("=" * 60)
        print(f"  DATASET SUMMARY  ({df.shape[0]:,} rows × {df.shape[1]} columns)")
        print("=" * 60)

        num_cols = df.select_dtypes(include=np.number).columns.tolist()
        cat_cols = df.select_dtypes(include="object").columns.tolist()

        print(f"\n  Numeric  columns ({len(num_cols)}): {num_cols}")
        print(f"  Categorical columns ({len(cat_cols)}): {cat_cols}")

        null_summary = df.isnull().sum()
        null_summary = null_summary[null_summary > 0]
        if not null_summary.empty:
            print("\n  Missing values (columns with nulls):")
            for col, count in null_summary.items():
                pct = count / len(df) * 100
                print(f"    {col:<30} {count:>6,}  ({pct:.1f} %)")
        else:
            print("\n  ✅  No missing values detected.")

        print(f"\n  First 20 rows preview:")
        try:
            from IPython.display import display  # type: ignore
            display(df.head(20))
        except ImportError:
            print(df.head(20).to_string())
        print("=" * 60)

    def handle_missing_values(
        self,
        strategy: str = "mean",
        columns: Optional[List[str]] = None,
        constant: Optional[Union[int, float, str]] = None,
    ) -> None:
        """
        Impute missing values in the working dataset.

        Parameters
        ----------
        strategy : {'mean', 'median', 'mode', 'constant'}
        columns  : list of str, optional
            Restrict imputation to these columns.  Default: all columns.
        constant : scalar, optional
            Required when ``strategy='constant'``.
        """
        self._require_data()
        targets = columns or self.df.columns.tolist()
        df = self.df

        for col in targets:
            if df[col].isnull().sum() == 0:
                continue
            if strategy == "mean":
                if pd.api.types.is_numeric_dtype(df[col]):
                    self.df[col] = df[col].fillna(df[col].mean())
            elif strategy == "median":
                if pd.api.types.is_numeric_dtype(df[col]):
                    self.df[col] = df[col].fillna(df[col].median())
            elif strategy == "mode":
                mode_val = df[col].mode()
                if not mode_val.empty:
                    self.df[col] = df[col].fillna(mode_val[0])
            elif strategy == "constant":
                if constant is None:
                    raise ValueError("Provide `constant` when strategy='constant'.")
                self.df[col] = df[col].fillna(constant)
            else:
                raise ValueError(f"Unknown strategy '{strategy}'. "
                                 "Choose from: mean, median, mode, constant.")

        remaining = df[targets].isnull().sum().sum()
        print(f"✅  Imputation complete ({strategy}). "
              f"Remaining nulls in targets: {remaining:,}")

    def remove_duplicates(self) -> None:
        """
        Drop exact duplicate rows from the working dataset, reporting
        how many were removed.
        """
        self._require_data()
        before = len(self.df)
        self.df.drop_duplicates(inplace=True)
        self.df.reset_index(drop=True, inplace=True)
        removed = before - len(self.df)
        print(f"✅  Removed {removed:,} duplicate row(s). "
              f"Dataset now has {len(self.df):,} rows.")

    def handle_outliers(
        self,
        columns: Optional[List[str]] = None,
        action: str = "flag",
        flag_column_suffix: str = "_outlier",
    ) -> None:
        """
        Detect outliers using the IQR fence method (1.5 × IQR).

        Parameters
        ----------
        columns : list of str, optional
            Numeric columns to inspect.  Defaults to all numeric columns.
        action  : {'flag', 'delete'}
            * ``'flag'``   – add a boolean column ``<col>_outlier``.
            * ``'delete'`` – drop all rows where any target column is an outlier.
        flag_column_suffix : str
            Suffix appended to the column name when ``action='flag'``.
        """
        self._require_data()
        num_cols = self.df.select_dtypes(include=np.number).columns.tolist()
        targets = columns or num_cols

        outlier_mask = pd.Series(False, index=self.df.index)

        for col in targets:
            if col not in self.df.columns:
                warnings.warn(f"Column '{col}' not found – skipped.")
                continue
            q1 = self.df[col].quantile(0.25)
            q3 = self.df[col].quantile(0.75)
            iqr = q3 - q1
            lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
            col_mask = (self.df[col] < lo) | (self.df[col] > hi)

            if action == "flag":
                self.df[f"{col}{flag_column_suffix}"] = col_mask
                print(f"  ⚑  {col}: {col_mask.sum():,} outlier(s) flagged.")
            elif action == "delete":
                outlier_mask |= col_mask
            else:
                raise ValueError("action must be 'flag' or 'delete'.")

        if action == "delete":
            before = len(self.df)
            self.df = self.df[~outlier_mask].reset_index(drop=True)
            print(f"✅  Deleted {before - len(self.df):,} outlier row(s).")

    def delete_rows(self, indices: Optional[str] = None) -> None:
        """
        Interactively delete rows by index.

        Parameters
        ----------
        indices : str, optional
            Comma-separated row indices, e.g. ``'0, 5, 12'``.  If *None*,
            the user is prompted for input.
        """
        self._require_data()
        if indices is None:
            indices = input("Enter comma-separated row indices to delete: ")
        idx_list = [int(i.strip()) for i in indices.split(",") if i.strip().isdigit()]
        before = len(self.df)
        self.df.drop(index=idx_list, inplace=True, errors="ignore")
        self.df.reset_index(drop=True, inplace=True)
        print(f"✅  Removed {before - len(self.df)} row(s).")

    def delete_columns(self, columns: Optional[str] = None) -> None:
        """
        Interactively delete columns by name.

        Parameters
        ----------
        columns : str, optional
            Comma-separated column names, e.g. ``'Age, Cabin'``.  If *None*,
            the user is prompted for input.
        """
        self._require_data()
        if columns is None:
            columns = input("Enter comma-separated column names to delete: ")
        col_list = [c.strip() for c in columns.split(",") if c.strip()]
        before = self.df.shape[1]
        self.df.drop(columns=col_list, inplace=True, errors="ignore")
        print(f"✅  Removed {before - self.df.shape[1]} column(s).")

    # ── 3. FEATURE ENGINEERING PREPARATION ────────────────────────────────────

    def extract_normalized_numeric_data(
        self, method: str = "minmax"
    ) -> pd.DataFrame:
        """
        Scale numeric columns.

        Parameters
        ----------
        method : {'minmax', 'standard', 'robust'}
            * ``'minmax'``   – scales to [0, 1].
            * ``'standard'`` – zero mean, unit variance (Z-score).
            * ``'robust'``   – median-centred, IQR-scaled (outlier-resistant).

        Returns
        -------
        pd.DataFrame
            Scaled numeric DataFrame (same index as ``self.df``).
        """
        self._require_data()
        self._require_sklearn()

        num_df = self.df.select_dtypes(include=np.number).copy()
        scalers = {
            "minmax": MinMaxScaler(),
            "standard": StandardScaler(),
            "robust": RobustScaler(),
        }
        if method not in scalers:
            raise ValueError(f"method must be one of {list(scalers)}")

        scaler = scalers[method]
        scaled = scaler.fit_transform(num_df.fillna(num_df.median()))
        result = pd.DataFrame(scaled, columns=num_df.columns, index=num_df.index)
        print(f"✅  Numeric data normalised with '{method}' scaling "
              f"({result.shape[1]} columns).")
        return result

    def extract_normalized_categorical_data(
        self, method: str = "onehot"
    ) -> pd.DataFrame:
        """
        Encode categorical columns.

        Parameters
        ----------
        method : {'onehot', 'ordinal', 'uniform'}
            * ``'onehot'``   – one-hot encoding (sparse=False).
            * ``'ordinal'``  – integer label encoding.
            * ``'uniform'``  – ordinal then min-max scaled to [0, 1].

        Returns
        -------
        pd.DataFrame
            Encoded categorical DataFrame.
        """
        self._require_data()
        self._require_sklearn()

        cat_df = self.df.select_dtypes(include="object").copy()
        if cat_df.empty:
            print("⚠️  No categorical columns found.")
            return pd.DataFrame(index=self.df.index)

        # fill NaN with mode for encoding
        for col in cat_df.columns:
            mode = cat_df[col].mode()
            if not mode.empty:
                cat_df[col] = cat_df[col].fillna(mode[0])

        if method == "onehot":
            enc = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
            encoded = enc.fit_transform(cat_df)
            cols = enc.get_feature_names_out(cat_df.columns)
            result = pd.DataFrame(encoded, columns=cols, index=cat_df.index)

        elif method in ("ordinal", "uniform"):
            enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
            encoded = enc.fit_transform(cat_df)
            result = pd.DataFrame(encoded, columns=cat_df.columns, index=cat_df.index)
            if method == "uniform":
                result = (result - result.min()) / (result.max() - result.min() + 1e-9)
        else:
            raise ValueError("method must be 'onehot', 'ordinal', or 'uniform'.")

        print(f"✅  Categorical data encoded with '{method}' "
              f"({result.shape[1]} output columns).")
        return result

    def get_merged_normalized_dataset(
        self,
        numeric_method: str = "standard",
        categorical_method: str = "onehot",
    ) -> pd.DataFrame:
        """
        Produce a unified DataFrame of scaled numeric + encoded categorical columns.

        Parameters
        ----------
        numeric_method : str
            Passed to :meth:`extract_normalized_numeric_data`.
        categorical_method : str
            Passed to :meth:`extract_normalized_categorical_data`.

        Returns
        -------
        pd.DataFrame
        """
        num = self.extract_normalized_numeric_data(method=numeric_method)
        cat = self.extract_normalized_categorical_data(method=categorical_method)
        merged = pd.concat([num, cat], axis=1)
        print(f"✅  Merged dataset: {merged.shape[0]:,} rows × {merged.shape[1]} columns.")
        return merged

    # ── 4. ADVANCED INTERACTIVE VISUALISATION ─────────────────────────────────

    def plot_numeric_univariate(self, column: str) -> None:
        """
        3-panel subplot for a single numeric column:
        left  – horizontal Violin / Box overlay
        centre – Scatter (Index vs Value)
        right  – Histogram with KDE overlay

        Parameters
        ----------
        column : str
            Name of a numeric column in ``self.df``.
        """
        self._require_data()
        self._require_plotly()

        if column not in self.df.columns:
            raise KeyError(f"Column '{column}' not found.")

        series = self.df[column].dropna()

        fig = make_subplots(
            rows=1, cols=3,
            subplot_titles=["Violin / Box", "Scatter (index vs value)", "Histogram"],
        )

        # Violin + Box
        fig.add_trace(go.Violin(
            x=series, name=column, box_visible=True,
            meanline_visible=True, orientation="h",
            line_color="#7c6af7", fillcolor="rgba(124,106,247,0.25)",
        ), row=1, col=1)

        # Scatter
        fig.add_trace(go.Scatter(
            x=series.index, y=series, mode="markers",
            marker=dict(size=4, color="#f97316", opacity=0.6),
            name=column,
        ), row=1, col=2)

        # Histogram
        fig.add_trace(go.Histogram(
            x=series, nbinsx=30,
            marker_color="#22d3ee", opacity=0.75,
            name=column,
        ), row=1, col=3)

        fig.update_layout(
            title_text=f"Univariate Analysis — {column}",
            showlegend=False,
            height=420,
            template="plotly_dark",
        )
        fig.show()

    def plot_relationship(self, col_x: str, col_y: str) -> None:
        """
        Smart bivariate chart that auto-selects the appropriate plot type.

        * Num–Num  → Scatter with OLS trendline
        * Cat–Num  → Box plot with overlaid points
        * Cat–Cat  → Grouped bar chart

        Parameters
        ----------
        col_x, col_y : str
            Column names in ``self.df``.
        """
        self._require_data()
        self._require_plotly()

        df = self.df[[col_x, col_y]].dropna()
        x_is_num = pd.api.types.is_numeric_dtype(df[col_x])
        y_is_num = pd.api.types.is_numeric_dtype(df[col_y])

        if x_is_num and y_is_num:
            fig = px.scatter(
                df, x=col_x, y=col_y, trendline="ols",
                title=f"Scatter: {col_x} vs {col_y} (OLS trendline)",
                template="plotly_dark", opacity=0.65,
                color_discrete_sequence=["#f97316"],
            )

        elif not x_is_num and y_is_num:
            fig = px.box(
                df, x=col_x, y=col_y, points="all",
                title=f"Box: {col_y} by {col_x}",
                template="plotly_dark",
                color=col_x,
            )

        elif x_is_num and not y_is_num:
            fig = px.box(
                df, x=col_y, y=col_x, points="all",
                title=f"Box: {col_x} by {col_y}",
                template="plotly_dark",
                color=col_y,
            )

        else:  # cat–cat
            ct = pd.crosstab(df[col_x], df[col_y]).reset_index().melt(id_vars=col_x)
            fig = px.bar(
                ct, x=col_x, y="value", color=col_y, barmode="group",
                title=f"Grouped Bar: {col_x} vs {col_y}",
                template="plotly_dark",
            )

        fig.update_layout(height=480)
        fig.show()

    def plot_categorical_frequency(self, column: str) -> None:
        """
        Bar chart of value counts for a categorical column, with percentage
        labels annotated on each bar.

        Parameters
        ----------
        column : str
            Name of a categorical column in ``self.df``.
        """
        self._require_data()
        self._require_plotly()

        counts = self.df[column].value_counts().reset_index()
        counts.columns = [column, "count"]
        counts["pct"] = (counts["count"] / counts["count"].sum() * 100).round(1)

        fig = px.bar(
            counts, x=column, y="count",
            text=counts["pct"].apply(lambda p: f"{p}%"),
            title=f"Frequency Distribution — {column}",
            template="plotly_dark",
            color="count",
            color_continuous_scale="Viridis",
        )
        fig.update_traces(textposition="outside")
        fig.update_layout(height=450, coloraxis_showscale=False)
        fig.show()

    # ── 5. DEEP STATISTICAL INSIGHTS ──────────────────────────────────────────

    def plot_all_associations_heatmap(self) -> None:
        """
        Unified association heatmap across all column pairs:

        * Numeric–Numeric   → Pearson's r  (range –1 … +1)
        * Cat–Cat           → Cramér's V   (range  0 … +1)
        * Numeric–Cat (mixed) → η² (Eta-squared via one-way ANOVA, range 0 … +1)

        The matrix is symmetric; the colour scale is centred at 0.
        """
        self._require_data()
        self._require_plotly()

        cols = self.df.columns.tolist()
        n = len(cols)
        matrix = np.zeros((n, n))

        num_set = set(self.df.select_dtypes(include=np.number).columns)

        for i, ci in enumerate(cols):
            for j, cj in enumerate(cols):
                if i == j:
                    matrix[i, j] = 1.0
                    continue
                if i > j:
                    matrix[i, j] = matrix[j, i]
                    continue

                si = self.df[ci].dropna()
                sj = self.df[cj].dropna()
                common = si.index.intersection(sj.index)
                si, sj = si[common], sj[common]

                if len(common) < 3:
                    matrix[i, j] = matrix[j, i] = 0.0
                    continue

                i_num = ci in num_set
                j_num = cj in num_set

                if i_num and j_num:
                    r, _ = stats.pearsonr(si, sj)
                    matrix[i, j] = matrix[j, i] = r
                elif not i_num and not j_num:
                    v = _cramers_v(si, sj)
                    matrix[i, j] = matrix[j, i] = v
                else:
                    cat_s, num_s = (si, sj) if not i_num else (sj, si)
                    eta = _eta_squared(cat_s, num_s)
                    matrix[i, j] = matrix[j, i] = eta

        hover = []
        for i, ci in enumerate(cols):
            row_hover = []
            for j, cj in enumerate(cols):
                i_num, j_num = ci in num_set, cj in num_set
                if i == j:
                    metric = "self"
                elif i_num and j_num:
                    metric = "Pearson r"
                elif not i_num and not j_num:
                    metric = "Cramér V"
                else:
                    metric = "Eta²"
                row_hover.append(f"{ci} × {cj}<br>{metric}: {matrix[i, j]:.3f}")
            hover.append(row_hover)

        fig = go.Figure(go.Heatmap(
            z=matrix,
            x=cols, y=cols,
            text=hover,
            hoverinfo="text",
            colorscale="RdBu",
            zmid=0,
            zmin=-1, zmax=1,
            colorbar=dict(title="Association"),
        ))
        fig.update_layout(
            title="Unified Association Heatmap  "
                  "(Pearson r | Cramér V | Eta²)",
            template="plotly_dark",
            height=max(500, 60 * n),
            width=max(550, 65 * n),
            xaxis=dict(tickangle=45),
        )
        fig.show()

    # ── helpers ───────────────────────────────────────────────────────────────

    def reset(self) -> None:
        """Restore the working dataset to the original loaded state."""
        if self._original_df is None:
            raise RuntimeError("No data loaded.")
        self.df = self._original_df.copy(deep=True)
        print("✅  Dataset reset to original.")

    def _require_data(self) -> None:
        if self.df is None:
            raise RuntimeError("No data loaded. Call upload_data() first.")

    @staticmethod
    def _require_plotly() -> None:
        if not PLOTLY_AVAILABLE:
            raise ImportError("Install plotly:  pip install plotly")

    @staticmethod
    def _require_sklearn() -> None:
        if not SKLEARN_AVAILABLE:
            raise ImportError("Install scikit-learn:  pip install scikit-learn")


# =============================================================================
# PlottingMethods
# =============================================================================

class PlottingMethods:
    """
    Granular chart helpers that each return an HTML string containing the
    Plotly figure for flexible embedding (e.g. in notebooks or Dash apps).

    All methods accept a ``pd.Series`` or a ``pd.DataFrame`` plus column names,
    and return ``str`` (the full HTML representation of the figure).

    Static methods – no instance state required.
    """

    @staticmethod
    def bar_chart(
        series: pd.Series,
        title: str = "Bar Chart",
        x_label: str = "Category",
        y_label: str = "Count",
        color: str = "#7c6af7",
        show_pct: bool = True,
    ) -> str:
        """
        Vertical bar chart with optional percentage labels.

        Parameters
        ----------
        series    : pd.Series – value-count series (index = categories, values = counts).
        title     : str
        x_label   : str
        y_label   : str
        color     : str        – hex / CSS color for bars.
        show_pct  : bool       – annotate bars with percentage of total.

        Returns
        -------
        str  – HTML string embedding the Plotly figure.
        """
        if not PLOTLY_AVAILABLE:
            raise ImportError("plotly is required.")
        if series is None or series.empty:
            return "<p>No data provided.</p>"

        total = series.sum()
        text = [f"{v / total * 100:.1f}%" for v in series] if show_pct else None

        fig = go.Figure(go.Bar(
            x=series.index.astype(str),
            y=series.values,
            text=text,
            textposition="outside",
            marker_color=color,
        ))
        fig.update_layout(
            title=title,
            xaxis_title=x_label,
            yaxis_title=y_label,
            template="plotly_dark",
            height=420,
        )
        return fig.to_html(full_html=False, include_plotlyjs="cdn")

    @staticmethod
    def pie_chart(
        series: pd.Series,
        title: str = "Pie Chart",
        hole: float = 0.0,
    ) -> str:
        """
        Pie (or donut) chart.

        Parameters
        ----------
        series : pd.Series – value-count series.
        title  : str
        hole   : float     – 0 = full pie; 0.4 = donut.

        Returns
        -------
        str  – HTML string.
        """
        if not PLOTLY_AVAILABLE:
            raise ImportError("plotly is required.")
        if series is None or series.empty:
            return "<p>No data provided.</p>"

        fig = go.Figure(go.Pie(
            labels=series.index.astype(str),
            values=series.values,
            hole=hole,
            textinfo="label+percent",
        ))
        fig.update_layout(title=title, template="plotly_dark", height=420)
        return fig.to_html(full_html=False, include_plotlyjs="cdn")

    @staticmethod
    def histogram(
        series: pd.Series,
        title: str = "Histogram",
        x_label: str = "Value",
        bins: int = 30,
        color: str = "#22d3ee",
        show_kde: bool = True,
    ) -> str:
        """
        Histogram with optional KDE curve overlay.

        Parameters
        ----------
        series   : pd.Series – numeric data.
        title    : str
        x_label  : str
        bins     : int
        color    : str
        show_kde : bool – overlay a Kernel Density Estimate curve.

        Returns
        -------
        str  – HTML string.
        """
        if not PLOTLY_AVAILABLE:
            raise ImportError("plotly is required.")
        if series is None or series.empty:
            return "<p>No data provided.</p>"

        clean = series.dropna()
        fig = go.Figure()

        fig.add_trace(go.Histogram(
            x=clean,
            nbinsx=bins,
            marker_color=color,
            opacity=0.75,
            name="Frequency",
            histnorm="probability density" if show_kde else "",
        ))

        if show_kde and len(clean) > 2:
            kde_x = np.linspace(clean.min(), clean.max(), 200)
            kde = stats.gaussian_kde(clean)
            fig.add_trace(go.Scatter(
                x=kde_x, y=kde(kde_x),
                mode="lines",
                line=dict(color="#f97316", width=2),
                name="KDE",
            ))

        fig.update_layout(
            title=title,
            xaxis_title=x_label,
            yaxis_title="Density" if show_kde else "Count",
            template="plotly_dark",
            height=420,
            barmode="overlay",
        )
        return fig.to_html(full_html=False, include_plotlyjs="cdn")


In [ ]:
"""
data_inspector.py
=================
Reusable Python classes for automated CSV data ingestion, cleaning,
feature engineering preparation, and interactive statistical visualization.

Classes
-------
DataInspector  – core EDA / cleaning / normalisation engine
PlottingMethods – granular chart helpers returning HTML-embeddable figures

Usage (Google Colab)
--------------------
    from data_inspector import DataInspector, PlottingMethods
    di = DataInspector()
    di.upload_data()          # triggers Colab file picker
    di.data_summary()
    di.handle_missing_values(strategy="median")
    di.plot_numeric_univariate("Age")
    di.plot_all_associations_heatmap()
"""

from __future__ import annotations

import io
import warnings
from typing import Dict, List, Optional, Tuple, Union

import numpy as np
import pandas as pd
from scipy import stats

# ── optional imports (graceful degradation) ──────────────────────────────────
try:
    import plotly.express as px
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
    PLOTLY_AVAILABLE = True
except ImportError:
    PLOTLY_AVAILABLE = False
    warnings.warn("plotly not installed – visualisation methods will be unavailable.")

try:
    from sklearn.preprocessing import (
        MinMaxScaler, StandardScaler, RobustScaler,
        OrdinalEncoder, OneHotEncoder,
    )
    SKLEARN_AVAILABLE = True
except ImportError:
    SKLEARN_AVAILABLE = False
    warnings.warn("scikit-learn not installed – normalisation methods will be unavailable.")

# ── garbage strings treated as NaN everywhere ─────────────────────────────────
_GARBAGE = {"?", "n/a", "na", "null", "none", "nan", "", " ", "--", "n.a.", "n.a", "unknown"}


# =============================================================================
# Helper utilities
# =============================================================================

def _cramers_v(x: pd.Series, y: pd.Series) -> float:
    """Cramér's V association statistic for two categorical series."""
    confusion = pd.crosstab(x, y)
    chi2 = stats.chi2_contingency(confusion, correction=False)[0]
    n = confusion.values.sum()
    phi2 = chi2 / n
    r, k = confusion.shape
    phi2corr = max(0.0, phi2 - ((k - 1) * (r - 1)) / (n - 1))
    rcorr = r - (r - 1) ** 2 / (n - 1)
    kcorr = k - (k - 1) ** 2 / (n - 1)
    denom = min(kcorr - 1, rcorr - 1)
    return 0.0 if denom <= 0 else np.sqrt(phi2corr / denom)


def _eta_squared(cat: pd.Series, num: pd.Series) -> float:
    """η² (Eta-squared) via one-way ANOVA – effect size for cat→num."""
    groups = [num[cat == lvl].dropna() for lvl in cat.unique()]
    groups = [g for g in groups if len(g) > 0]
    if len(groups) < 2:
        return 0.0
    f, p = stats.f_oneway(*groups)
    if np.isnan(f):
        return 0.0
    grand_mean = num.mean()
    ss_between = sum(len(g) * (g.mean() - grand_mean) ** 2 for g in groups)
    ss_total = ((num - grand_mean) ** 2).sum()
    return 0.0 if ss_total == 0 else ss_between / ss_total


# =============================================================================
# DataInspector
# =============================================================================

class DataInspector:
    """
    End-to-end tool for CSV data ingestion, cleaning, feature engineering
    preparation, and interactive statistical visualisation.

    Attributes
    ----------
    df : pd.DataFrame
        The working dataset (mutated in-place by cleaning methods).
    _original_df : pd.DataFrame
        Immutable snapshot taken at load time.
    """

    # ── construction ──────────────────────────────────────────────────────────

    def __init__(self) -> None:
        self.df: Optional[pd.DataFrame] = None
        self._original_df: Optional[pd.DataFrame] = None

    # ── 1. DATA INGESTION & SANITISATION ──────────────────────────────────────

    def upload_data(self, filepath: Optional[str] = None) -> None:
        """
        Load a CSV into ``self.df``.

        Parameters
        ----------
        filepath : str, optional
            Absolute or relative path to a CSV file.  When *None* and the
            environment is Google Colab, the Colab file-picker widget is
            launched automatically.

        Side-effects
        ------------
        * Replaces garbage strings with ``np.nan``.
        * Force-converts columns to numeric where possible.
        * Stores an immutable copy in ``self._original_df``.
        """
        if filepath is None:
            try:
                from google.colab import files  # type: ignore
                print("📂  Please select a CSV file …")
                uploaded = files.upload()
                name = next(iter(uploaded))
                raw = io.BytesIO(uploaded[name])
            except (ImportError, StopIteration):
                raise ValueError(
                    "No filepath supplied and not running in Colab. "
                    "Pass filepath='/path/to/file.csv'."
                )
        else:
            raw = filepath

        df = pd.read_csv(raw, na_values=list(_GARBAGE), keep_default_na=True)

        # secondary pass: lower-strip strings not caught by na_values
        for col in df.select_dtypes(include="object").columns:
            df[col] = df[col].apply(
                lambda v: np.nan
                if isinstance(v, str) and v.strip().lower() in _GARBAGE
                else v
            )

        # auto-type correction
        df = self._auto_type_correct(df)

        self.df = df
        self._original_df = df.copy(deep=True)
        print(f"✅  Loaded {df.shape[0]:,} rows × {df.shape[1]} columns.")

    @staticmethod
    def _auto_type_correct(df: pd.DataFrame) -> pd.DataFrame:
        """
        Attempt to coerce object columns to numeric.  A column is only
        converted if the result is **not** entirely null (preserves columns
        that are genuinely categorical).
        """
        for col in df.select_dtypes(include="object").columns:
            trial = pd.to_numeric(df[col], errors="coerce")
            if trial.notna().any():
                df[col] = trial
        return df

    # ── 2. STRUCTURAL ANALYSIS & CLEANING ─────────────────────────────────────

    def data_summary(self) -> None:
        """
        Print a concise structural report:
        * row / column counts
        * first 20 rows preview
        * numerical vs categorical breakdown
        * per-column null counts and dtypes
        """
        self._require_data()
        df = self.df

        print("=" * 60)
        print(f"  DATASET SUMMARY  ({df.shape[0]:,} rows × {df.shape[1]} columns)")
        print("=" * 60)

        num_cols = df.select_dtypes(include=np.number).columns.tolist()
        cat_cols = df.select_dtypes(include="object").columns.tolist()

        print(f"\n  Numeric  columns ({len(num_cols)}): {num_cols}")
        print(f"  Categorical columns ({len(cat_cols)}): {cat_cols}")

        null_summary = df.isnull().sum()
        null_summary = null_summary[null_summary > 0]
        if not null_summary.empty:
            print("\n  Missing values (columns with nulls):")
            for col, count in null_summary.items():
                pct = count / len(df) * 100
                print(f"    {col:<30} {count:>6,}  ({pct:.1f} %)")
        else:
            print("\n  ✅  No missing values detected.")

        print(f"\n  First 20 rows preview:")
        try:
            from IPython.display import display  # type: ignore
            display(df.head(20))
        except ImportError:
            print(df.head(20).to_string())
        print("=" * 60)

    def handle_missing_values(
        self,
        strategy: str = "mean",
        columns: Optional[List[str]] = None,
        constant: Optional[Union[int, float, str]] = None,
    ) -> None:
        """
        Impute missing values in the working dataset.

        Parameters
        ----------
        strategy : {'mean', 'median', 'mode', 'constant'}
        columns  : list of str, optional
            Restrict imputation to these columns.  Default: all columns.
        constant : scalar, optional
            Required when ``strategy='constant'``.
        """
        self._require_data()
        targets = columns or self.df.columns.tolist()
        df = self.df

        for col in targets:
            if df[col].isnull().sum() == 0:
                continue
            if strategy == "mean":
                if pd.api.types.is_numeric_dtype(df[col]):
                    self.df[col] = df[col].fillna(df[col].mean())
            elif strategy == "median":
                if pd.api.types.is_numeric_dtype(df[col]):
                    self.df[col] = df[col].fillna(df[col].median())
            elif strategy == "mode":
                mode_val = df[col].mode()
                if not mode_val.empty:
                    self.df[col] = df[col].fillna(mode_val[0])
            elif strategy == "constant":
                if constant is None:
                    raise ValueError("Provide `constant` when strategy='constant'.")
                self.df[col] = df[col].fillna(constant)
            else:
                raise ValueError(f"Unknown strategy '{strategy}'. "
                                 "Choose from: mean, median, mode, constant.")

        remaining = df[targets].isnull().sum().sum()
        print(f"✅  Imputation complete ({strategy}). "
              f"Remaining nulls in targets: {remaining:,}")

    def remove_duplicates(self) -> None:
        """
        Drop exact duplicate rows from the working dataset, reporting
        how many were removed.
        """
        self._require_data()
        before = len(self.df)
        self.df.drop_duplicates(inplace=True)
        self.df.reset_index(drop=True, inplace=True)
        removed = before - len(self.df)
        print(f"✅  Removed {removed:,} duplicate row(s). "
              f"Dataset now has {len(self.df):,} rows.")

    def handle_outliers(
        self,
        columns: Optional[List[str]] = None,
        action: str = "flag",
        flag_column_suffix: str = "_outlier",
    ) -> None:
        """
        Detect outliers using the IQR fence method (1.5 × IQR).

        Parameters
        ----------
        columns : list of str, optional
            Numeric columns to inspect.  Defaults to all numeric columns.
        action  : {'flag', 'delete'}
            * ``'flag'``   – add a boolean column ``<col>_outlier``.
            * ``'delete'`` – drop all rows where any target column is an outlier.
        flag_column_suffix : str
            Suffix appended to the column name when ``action='flag'``.
        """
        self._require_data()
        num_cols = self.df.select_dtypes(include=np.number).columns.tolist()
        targets = columns or num_cols

        outlier_mask = pd.Series(False, index=self.df.index)

        for col in targets:
            if col not in self.df.columns:
                warnings.warn(f"Column '{col}' not found – skipped.")
                continue
            q1 = self.df[col].quantile(0.25)
            q3 = self.df[col].quantile(0.75)
            iqr = q3 - q1
            lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
            col_mask = (self.df[col] < lo) | (self.df[col] > hi)

            if action == "flag":
                self.df[f"{col}{flag_column_suffix}"] = col_mask
                print(f"  ⚑  {col}: {col_mask.sum():,} outlier(s) flagged.")
            elif action == "delete":
                outlier_mask |= col_mask
            else:
                raise ValueError("action must be 'flag' or 'delete'.")

        if action == "delete":
            before = len(self.df)
            self.df = self.df[~outlier_mask].reset_index(drop=True)
            print(f"✅  Deleted {before - len(self.df):,} outlier row(s).")

    def delete_rows(self, indices: Optional[str] = None) -> None:
        """
        Interactively delete rows by index.

        Parameters
        ----------
        indices : str, optional
            Comma-separated row indices, e.g. ``'0, 5, 12'``.  If *None*,
            the user is prompted for input.
        """
        self._require_data()
        if indices is None:
            indices = input("Enter comma-separated row indices to delete: ")
        idx_list = [int(i.strip()) for i in indices.split(",") if i.strip().isdigit()]
        before = len(self.df)
        self.df.drop(index=idx_list, inplace=True, errors="ignore")
        self.df.reset_index(drop=True, inplace=True)
        print(f"✅  Removed {before - len(self.df)} row(s).")

    def delete_columns(self, columns: Optional[str] = None) -> None:
        """
        Interactively delete columns by name.

        Parameters
        ----------
        columns : str, optional
            Comma-separated column names, e.g. ``'Age, Cabin'``.  If *None*,
            the user is prompted for input.
        """
        self._require_data()
        if columns is None:
            columns = input("Enter comma-separated column names to delete: ")
        col_list = [c.strip() for c in columns.split(",") if c.strip()]
        before = self.df.shape[1]
        self.df.drop(columns=col_list, inplace=True, errors="ignore")
        print(f"✅  Removed {before - self.df.shape[1]} column(s).")

    # ── 3. FEATURE ENGINEERING PREPARATION ────────────────────────────────────

    def extract_normalized_numeric_data(
        self, method: str = "minmax"
    ) -> pd.DataFrame:
        """
        Scale numeric columns.

        Parameters
        ----------
        method : {'minmax', 'standard', 'robust'}
            * ``'minmax'``   – scales to [0, 1].
            * ``'standard'`` – zero mean, unit variance (Z-score).
            * ``'robust'``   – median-centred, IQR-scaled (outlier-resistant).

        Returns
        -------
        pd.DataFrame
            Scaled numeric DataFrame (same index as ``self.df``).
        """
        self._require_data()
        self._require_sklearn()

        num_df = self.df.select_dtypes(include=np.number).copy()
        scalers = {
            "minmax": MinMaxScaler(),
            "standard": StandardScaler(),
            "robust": RobustScaler(),
        }
        if method not in scalers:
            raise ValueError(f"method must be one of {list(scalers)}")

        scaler = scalers[method]
        scaled = scaler.fit_transform(num_df.fillna(num_df.median()))
        result = pd.DataFrame(scaled, columns=num_df.columns, index=num_df.index)
        print(f"✅  Numeric data normalised with '{method}' scaling "
              f"({result.shape[1]} columns).")
        return result

    def extract_normalized_categorical_data(
        self, method: str = "onehot"
    ) -> pd.DataFrame:
        """
        Encode categorical columns.

        Parameters
        ----------
        method : {'onehot', 'ordinal', 'uniform'}
            * ``'onehot'``   – one-hot encoding (sparse=False).
            * ``'ordinal'``  – integer label encoding.
            * ``'uniform'``  – ordinal then min-max scaled to [0, 1].

        Returns
        -------
        pd.DataFrame
            Encoded categorical DataFrame.
        """
        self._require_data()
        self._require_sklearn()

        cat_df = self.df.select_dtypes(include="object").copy()
        if cat_df.empty:
            print("⚠️  No categorical columns found.")
            return pd.DataFrame(index=self.df.index)

        # fill NaN with mode for encoding
        for col in cat_df.columns:
            mode = cat_df[col].mode()
            if not mode.empty:
                cat_df[col] = cat_df[col].fillna(mode[0])

        if method == "onehot":
            enc = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
            encoded = enc.fit_transform(cat_df)
            cols = enc.get_feature_names_out(cat_df.columns)
            result = pd.DataFrame(encoded, columns=cols, index=cat_df.index)

        elif method in ("ordinal", "uniform"):
            enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
            encoded = enc.fit_transform(cat_df)
            result = pd.DataFrame(encoded, columns=cat_df.columns, index=cat_df.index)
            if method == "uniform":
                result = (result - result.min()) / (result.max() - result.min() + 1e-9)
        else:
            raise ValueError("method must be 'onehot', 'ordinal', or 'uniform'.")

        print(f"✅  Categorical data encoded with '{method}' "
              f"({result.shape[1]} output columns).")
        return result

    def get_merged_normalized_dataset(
        self,
        numeric_method: str = "standard",
        categorical_method: str = "onehot",
    ) -> pd.DataFrame:
        """
        Produce a unified DataFrame of scaled numeric + encoded categorical columns.

        Parameters
        ----------
        numeric_method : str
            Passed to :meth:`extract_normalized_numeric_data`.
        categorical_method : str
            Passed to :meth:`extract_normalized_categorical_data`.

        Returns
        -------
        pd.DataFrame
        """
        num = self.extract_normalized_numeric_data(method=numeric_method)
        cat = self.extract_normalized_categorical_data(method=categorical_method)
        merged = pd.concat([num, cat], axis=1)
        print(f"✅  Merged dataset: {merged.shape[0]:,} rows × {merged.shape[1]} columns.")
        return merged

    # ── 4. ADVANCED INTERACTIVE VISUALISATION ─────────────────────────────────

    def plot_numeric_univariate(self, column: str) -> None:
        """
        3-panel subplot for a single numeric column:
        left  – horizontal Violin / Box overlay
        centre – Scatter (Index vs Value)
        right  – Histogram with KDE overlay

        Parameters
        ----------
        column : str
            Name of a numeric column in ``self.df``.
        """
        self._require_data()
        self._require_plotly()

        if column not in self.df.columns:
            raise KeyError(f"Column '{column}' not found.")

        series = self.df[column].dropna()

        fig = make_subplots(
            rows=1, cols=3,
            subplot_titles=["Violin / Box", "Scatter (index vs value)", "Histogram"],
        )

        # Violin + Box
        fig.add_trace(go.Violin(
            x=series, name=column, box_visible=True,
            meanline_visible=True, orientation="h",
            line_color="#7c6af7", fillcolor="rgba(124,106,247,0.25)",
        ), row=1, col=1)

        # Scatter
        fig.add_trace(go.Scatter(
            x=series.index, y=series, mode="markers",
            marker=dict(size=4, color="#f97316", opacity=0.6),
            name=column,
        ), row=1, col=2)

        # Histogram
        fig.add_trace(go.Histogram(
            x=series, nbinsx=30,
            marker_color="#22d3ee", opacity=0.75,
            name=column,
        ), row=1, col=3)

        fig.update_layout(
            title_text=f"Univariate Analysis — {column}",
            showlegend=False,
            height=420,
            template="plotly_dark",
        )
        fig.show()

    def plot_relationship(self, col_x: str, col_y: str) -> None:
        """
        Smart bivariate chart that auto-selects the appropriate plot type.

        * Num–Num  → Scatter with OLS trendline
        * Cat–Num  → Box plot with overlaid points
        * Cat–Cat  → Grouped bar chart

        Parameters
        ----------
        col_x, col_y : str
            Column names in ``self.df``.
        """
        self._require_data()
        self._require_plotly()

        df = self.df[[col_x, col_y]].dropna()
        x_is_num = pd.api.types.is_numeric_dtype(df[col_x])
        y_is_num = pd.api.types.is_numeric_dtype(df[col_y])

        if x_is_num and y_is_num:
            fig = px.scatter(
                df, x=col_x, y=col_y, trendline="ols",
                title=f"Scatter: {col_x} vs {col_y} (OLS trendline)",
                template="plotly_dark", opacity=0.65,
                color_discrete_sequence=["#f97316"],
            )

        elif not x_is_num and y_is_num:
            fig = px.box(
                df, x=col_x, y=col_y, points="all",
                title=f"Box: {col_y} by {col_x}",
                template="plotly_dark",
                color=col_x,
            )

        elif x_is_num and not y_is_num:
            fig = px.box(
                df, x=col_y, y=col_x, points="all",
                title=f"Box: {col_x} by {col_y}",
                template="plotly_dark",
                color=col_y,
            )

        else:  # cat–cat
            ct = pd.crosstab(df[col_x], df[col_y]).reset_index().melt(id_vars=col_x)
            fig = px.bar(
                ct, x=col_x, y="value", color=col_y, barmode="group",
                title=f"Grouped Bar: {col_x} vs {col_y}",
                template="plotly_dark",
            )

        fig.update_layout(height=480)
        fig.show()

    def plot_categorical_frequency(self, column: str) -> None:
        """
        Bar chart of value counts for a categorical column, with percentage
        labels annotated on each bar.

        Parameters
        ----------
        column : str
            Name of a categorical column in ``self.df``.
        """
        self._require_data()
        self._require_plotly()

        counts = self.df[column].value_counts().reset_index()
        counts.columns = [column, "count"]
        counts["pct"] = (counts["count"] / counts["count"].sum() * 100).round(1)

        fig = px.bar(
            counts, x=column, y="count",
            text=counts["pct"].apply(lambda p: f"{p}%"),
            title=f"Frequency Distribution — {column}",
            template="plotly_dark",
            color="count",
            color_continuous_scale="Viridis",
        )
        fig.update_traces(textposition="outside")
        fig.update_layout(height=450, coloraxis_showscale=False)
        fig.show()

    # ── 5. DEEP STATISTICAL INSIGHTS ──────────────────────────────────────────

    def plot_all_associations_heatmap(self) -> None:
        """
        Unified association heatmap across all column pairs:

        * Numeric–Numeric   → Pearson's r  (range –1 … +1)
        * Cat–Cat           → Cramér's V   (range  0 … +1)
        * Numeric–Cat (mixed) → η² (Eta-squared via one-way ANOVA, range 0 … +1)

        The matrix is symmetric; the colour scale is centred at 0.
        """
        self._require_data()
        self._require_plotly()

        cols = self.df.columns.tolist()
        n = len(cols)
        matrix = np.zeros((n, n))

        num_set = set(self.df.select_dtypes(include=np.number).columns)

        for i, ci in enumerate(cols):
            for j, cj in enumerate(cols):
                if i == j:
                    matrix[i, j] = 1.0
                    continue
                if i > j:
                    matrix[i, j] = matrix[j, i]
                    continue

                si = self.df[ci].dropna()
                sj = self.df[cj].dropna()
                common = si.index.intersection(sj.index)
                si, sj = si[common], sj[common]

                if len(common) < 3:
                    matrix[i, j] = matrix[j, i] = 0.0
                    continue

                i_num = ci in num_set
                j_num = cj in num_set

                if i_num and j_num:
                    r, _ = stats.pearsonr(si, sj)
                    matrix[i, j] = matrix[j, i] = r
                elif not i_num and not j_num:
                    v = _cramers_v(si, sj)
                    matrix[i, j] = matrix[j, i] = v
                else:
                    cat_s, num_s = (si, sj) if not i_num else (sj, si)
                    eta = _eta_squared(cat_s, num_s)
                    matrix[i, j] = matrix[j, i] = eta

        hover = []
        for i, ci in enumerate(cols):
            row_hover = []
            for j, cj in enumerate(cols):
                i_num, j_num = ci in num_set, cj in num_set
                if i == j:
                    metric = "self"
                elif i_num and j_num:
                    metric = "Pearson r"
                elif not i_num and not j_num:
                    metric = "Cramér V"
                else:
                    metric = "Eta²"
                row_hover.append(f"{ci} × {cj}<br>{metric}: {matrix[i, j]:.3f}")
            hover.append(row_hover)

        fig = go.Figure(go.Heatmap(
            z=matrix,
            x=cols, y=cols,
            text=hover,
            hoverinfo="text",
            colorscale="RdBu",
            zmid=0,
            zmin=-1, zmax=1,
            colorbar=dict(title="Association"),
        ))
        fig.update_layout(
            title="Unified Association Heatmap  "
                  "(Pearson r | Cramér V | Eta²)",
            template="plotly_dark",
            height=max(500, 60 * n),
            width=max(550, 65 * n),
            xaxis=dict(tickangle=45),
        )
        fig.show()

    # ── helpers ───────────────────────────────────────────────────────────────

    def reset(self) -> None:
        """Restore the working dataset to the original loaded state."""
        if self._original_df is None:
            raise RuntimeError("No data loaded.")
        self.df = self._original_df.copy(deep=True)
        print("✅  Dataset reset to original.")

    def _require_data(self) -> None:
        if self.df is None:
            raise RuntimeError("No data loaded. Call upload_data() first.")

    @staticmethod
    def _require_plotly() -> None:
        if not PLOTLY_AVAILABLE:
            raise ImportError("Install plotly:  pip install plotly")

    @staticmethod
    def _require_sklearn() -> None:
        if not SKLEARN_AVAILABLE:
            raise ImportError("Install scikit-learn:  pip install scikit-learn")


# =============================================================================
# PlottingMethods
# =============================================================================

class PlottingMethods:
    """
    Granular chart helpers that each return an HTML string containing the
    Plotly figure for flexible embedding (e.g. in notebooks or Dash apps).

    All methods accept a ``pd.Series`` or a ``pd.DataFrame`` plus column names,
    and return ``str`` (the full HTML representation of the figure).

    Static methods – no instance state required.
    """

    @staticmethod
    def bar_chart(
        series: pd.Series,
        title: str = "Bar Chart",
        x_label: str = "Category",
        y_label: str = "Count",
        color: str = "#7c6af7",
        show_pct: bool = True,
    ) -> str:
        """
        Vertical bar chart with optional percentage labels.

        Parameters
        ----------
        series    : pd.Series – value-count series (index = categories, values = counts).
        title     : str
        x_label   : str
        y_label   : str
        color     : str        – hex / CSS color for bars.
        show_pct  : bool       – annotate bars with percentage of total.

        Returns
        -------
        str  – HTML string embedding the Plotly figure.
        """
        if not PLOTLY_AVAILABLE:
            raise ImportError("plotly is required.")
        if series is None or series.empty:
            return "<p>No data provided.</p>"

        total = series.sum()
        text = [f"{v / total * 100:.1f}%" for v in series] if show_pct else None

        fig = go.Figure(go.Bar(
            x=series.index.astype(str),
            y=series.values,
            text=text,
            textposition="outside",
            marker_color=color,
        ))
        fig.update_layout(
            title=title,
            xaxis_title=x_label,
            yaxis_title=y_label,
            template="plotly_dark",
            height=420,
        )
        return fig.to_html(full_html=False, include_plotlyjs="cdn")

    @staticmethod
    def pie_chart(
        series: pd.Series,
        title: str = "Pie Chart",
        hole: float = 0.0,
    ) -> str:
        """
        Pie (or donut) chart.

        Parameters
        ----------
        series : pd.Series – value-count series.
        title  : str
        hole   : float     – 0 = full pie; 0.4 = donut.

        Returns
        -------
        str  – HTML string.
        """
        if not PLOTLY_AVAILABLE:
            raise ImportError("plotly is required.")
        if series is None or series.empty:
            return "<p>No data provided.</p>"

        fig = go.Figure(go.Pie(
            labels=series.index.astype(str),
            values=series.values,
            hole=hole,
            textinfo="label+percent",
        ))
        fig.update_layout(title=title, template="plotly_dark", height=420)
        return fig.to_html(full_html=False, include_plotlyjs="cdn")

    @staticmethod
    def histogram(
        series: pd.Series,
        title: str = "Histogram",
        x_label: str = "Value",
        bins: int = 30,
        color: str = "#22d3ee",
        show_kde: bool = True,
    ) -> str:
        """
        Histogram with optional KDE curve overlay.

        Parameters
        ----------
        series   : pd.Series – numeric data.
        title    : str
        x_label  : str
        bins     : int
        color    : str
        show_kde : bool – overlay a Kernel Density Estimate curve.

        Returns
        -------
        str  – HTML string.
        """
        if not PLOTLY_AVAILABLE:
            raise ImportError("plotly is required.")
        if series is None or series.empty:
            return "<p>No data provided.</p>"

        clean = series.dropna()
        fig = go.Figure()

        fig.add_trace(go.Histogram(
            x=clean,
            nbinsx=bins,
            marker_color=color,
            opacity=0.75,
            name="Frequency",
            histnorm="probability density" if show_kde else "",
        ))

        if show_kde and len(clean) > 2:
            kde_x = np.linspace(clean.min(), clean.max(), 200)
            kde = stats.gaussian_kde(clean)
            fig.add_trace(go.Scatter(
                x=kde_x, y=kde(kde_x),
                mode="lines",
                line=dict(color="#f97316", width=2),
                name="KDE",
            ))

        fig.update_layout(
            title=title,
            xaxis_title=x_label,
            yaxis_title="Density" if show_kde else "Count",
            template="plotly_dark",
            height=420,
            barmode="overlay",
        )
        return fig.to_html(full_html=False, include_plotlyjs="cdn")
